# ProgettoSNR — Rilevamento Malware + Offuscamento

Notebook unico, organizzato per **sezioni indipendenti**. Ogni sezione ha:
premessa markdown → cella codice.

**Struttura:**
1. Preparazione TFRecord (una tantum)
2. **BASE CONDIVISA** — config, dataset, architettura CNN, funzioni comuni a tutti i modelli
3. Modello CNN (EfficientNetB0 multitask) — training e test
4. Modello Isolation Forest su embedding — training e test
5. Modello VAE su embedding — training e test
6. Riepilogo finale — training + test in sequenza per tutti i modelli

**Struttura cartelle di output (per ogni modello):**

```
BASE_DIR/
├── cnn_model/
│   ├── model/     ← best_model.keras, scaler, labeler, feature_cols.json
│   ├── train/     ← plot e metriche del training/validation
│   └── test/      ← plot e metriche del test
├── if_model/
│   ├── model/     ← iso_embedding.pkl, if_calibration.json
│   ├── train/     ← plot di calibrazione sul train
│   └── test/      ← plot e CSV del test
└── vae_model/
    ├── model/     ← vae_encoder.keras, vae_decoder.keras, vae_calibration.json
    ├── train/     ← plot del training/validation
    └── test/      ← plot e CSV del test
```

**Come eseguire su Colab:** esegui le celle **in ordine, dall'alto verso il basso**, una sola volta ciascuna. La sezione 2 (BASE CONDIVISA) va sempre eseguita per prima in ogni sessione, perché tutte le sezioni successive dipendono dalle sue funzioni e variabili — se riavvii il runtime, ri-eseguila prima di riprendere da dove eri.

**Nota sulla revisione:** rispetto alla versione precedente sono state rimosse due celle di codice finali scollegate dal resto del notebook (un'architettura CNN alternativa con Gated Residual Network e cross-attention, mai richiamata altrove, con un test di salvataggio duplicato). Al loro posto è stata aggiunta la sezione 6, che esegue **training e test di tutti e tre i modelli** in un'unica cella.

---
## 1. Conversione PNG → TFRecord

Da eseguire **una volta sola** in assoluto (non ad ogni sessione). Legge le immagini PNG e i CSV di feature (`dataset_train.csv`, `dataset_val.csv`, `dataset_test.csv`) da Drive e produce `train.tfrecord`, `val.tfrecord`, `test.tfrecord` dentro `tfrecords/`, insieme a `meta.json` con l'elenco delle feature CSV usate.

Se i file `.tfrecord` sono già presenti, la cella salta la conversione per quello split.

Tutte le sezioni successive leggono da questi TFRecord: se il dataset cambia, cancella i vecchi `.tfrecord` da Drive e ri-esegui questa cella.

In [ ]:
# ==============================================================
# SEZIONE 1 — Conversione PNG → TFRecord
# Esegui UNA VOLTA SOLA. Produce train.tfrecord, val.tfrecord,
# test.tfrecord in BASE_DIR. Le sezioni successive leggono quelli.
# ==============================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import drive
from tqdm.auto import tqdm

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/ProgettoSNR"
IMG_DIR  = os.path.join(BASE_DIR, "dataset_images")
IMG_SIZE = 200

SPLITS = {
    "train": os.path.join(BASE_DIR, "dataset_train.csv"),
    "val":   os.path.join(BASE_DIR, "dataset_val.csv"),
    "test":  os.path.join(BASE_DIR, "dataset_test.csv"),
}

TFRECORD_DIR = os.path.join(BASE_DIR, "tfrecords")
os.makedirs(TFRECORD_DIR, exist_ok=True)


def _bytes(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _float_list(value):
    return tf.train.Feature(float_list=tf.train.FloatList(value=value))

def _int64(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))


def encode_example(img_path, csv_feat, label, filename):
    raw = tf.io.read_file(img_path)
    img = tf.io.decode_image(raw, channels=1, expand_animations=False)
    img.set_shape([None, None, 1])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    img_bytes = tf.io.serialize_tensor(img).numpy()

    feature = {
        "image":    _bytes(img_bytes),
        "csv_feat": _float_list(csv_feat.tolist()),
        "label":    _int64(int(label)),
        "filename": _bytes(filename.encode()),
    }
    return tf.train.Example(features=tf.train.Features(feature=feature))


def convert_split(split_name, csv_path, feature_cols):
    out_path = os.path.join(TFRECORD_DIR, f"{split_name}.tfrecord")
    if os.path.exists(out_path):
        print(f"{split_name}.tfrecord già presente, salto.")
        return feature_cols

    df = pd.read_csv(csv_path)
    if feature_cols is None:
        exclude = {"filename", "image_name", "label"}
        feature_cols = [c for c in df.columns if c not in exclude]

    print(f"Conversione {split_name}: {len(df)} campioni → {out_path}")
    errors = 0

    with tf.io.TFRecordWriter(out_path) as writer:
        for i, row in df.iterrows():
            img_path = os.path.join(IMG_DIR, row["image_name"])
            if not os.path.exists(img_path):
                print(f"  WARN: immagine mancante {img_path}, salto.")
                errors += 1
                continue
            try:
                feat = row[feature_cols].values.astype(np.float32)
                ex = encode_example(img_path, feat, row["label"], row["filename"])
                writer.write(ex.SerializeToString())
            except Exception as e:
                print(f"  WARN: errore su {row['filename']}: {e}")
                errors += 1

            if (i + 1) % 500 == 0:
                print(f"  {i + 1}/{len(df)} convertiti...")

    print(f"  Completato. Errori: {errors}")
    return feature_cols


# Esegui la conversione per tutti e tre i set.
# feature_cols derivato dal train CSV e riusato per val/test
# per garantire allineamento colonne.
feature_cols = None
for name, csv_path in SPLITS.items():
    feature_cols = convert_split(name, csv_path, feature_cols)

# Salva feature_cols per le sezioni successive
meta_path = os.path.join(TFRECORD_DIR, "meta.json")
with open(meta_path, "w") as f:
    json.dump({"feature_cols": feature_cols, "img_size": IMG_SIZE}, f)

print("=" * 60)
print(f"TFRecord pronti in: {TFRECORD_DIR}")
print(f"Feature CSV: {len(feature_cols)}")
print("Puoi ora eseguire la sezione 2 (BASE CONDIVISA).")
print("=" * 60)

---
## 2. Base condivisa

**Esegui questa cella prima di qualsiasi altra sezione (2–6), e ri-eseguila se riavvii il runtime.**

Contiene tutto ciò che CNN, Isolation Forest e VAE hanno in comune, così ogni modello non lo ridefinisce per conto suo:

- **Config**: percorsi su Drive, con una directory dedicata per ciascun modello (`cnn_model/`, `if_model/`, `vae_model/`), ognuna con le sottocartelle `model/` (pesi e artefatti), `train/` (plot di training) e `test/` (plot e CSV di test).
- **GPU / mixed precision**: rilevamento automatico.
- **Obfuscation labeler**: un Isolation Forest "di servizio" che, sulla base di feature di entropia del CSV, etichetta ogni sample come offuscato/non offuscato — questa etichetta (`is_obfuscated`) è il target usato per valutare sia il CNN sia i due modelli non supervisionati (IF ed VAE su embedding).
- **`make_dataset`**: costruisce la pipeline `tf.data` da TFRecord (immagine + feature CSV scalate + doppia etichetta malware/offuscamento).
- **Architettura CNN** (`build_malware_cnn`, `build_csv_branch`): EfficientNetB0 + ramo MLP residuale sulle feature CSV, fusione con gating, due teste (malware, offuscamento).
- **`load_trained_cnn`**: ricarica un CNN già addestrato dai soli pesi (bypassa un bug del loader Keras dovuto ai due pesi extra di uncertainty weighting).
- **`extract_fusion_embeddings`**: estrae l'embedding fuso (pre-testa) dal CNN — è l'input sia dell'Isolation Forest sia del VAE.
- **Predizioni e report**: `get_predictions`, `print_classification_report`, `save_plots`, `plot_confusion_matrix` — usate dal test di ogni modello.

In [ ]:
# ==============================================================
# SEZIONE 2 — BASE CONDIVISA
# Setup, config, funzioni comuni a CNN, Isolation Forest, VAE
# ==============================================================

import json, os, time, joblib, zipfile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.ensemble import IsolationForest
from sklearn.metrics import auc, confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models, optimizers
from google.colab import drive

drive.mount("/content/drive")
AUTOTUNE = tf.data.AUTOTUNE

# ==============================================================
# CONFIGURAZIONE
# ==============================================================

BASE_DIR = "/content/drive/MyDrive/ProgettoSNR"
TFRECORD_DIR = os.path.join(BASE_DIR, "tfrecords")

# --- Directory per modello: <modello>/{model,train,test} ---
CNN_DIR = os.path.join(BASE_DIR, "cnn_model")
IF_DIR  = os.path.join(BASE_DIR, "if_model")
VAE_DIR = os.path.join(BASE_DIR, "vae_model")

CNN_MODEL_DIR = os.path.join(CNN_DIR, "model")
CNN_TRAIN_DIR = os.path.join(CNN_DIR, "train")
CNN_TEST_DIR  = os.path.join(CNN_DIR, "test")

IF_MODEL_DIR = os.path.join(IF_DIR, "model")
IF_TRAIN_DIR = os.path.join(IF_DIR, "train")
IF_TEST_DIR  = os.path.join(IF_DIR, "test")

VAE_MODEL_DIR = os.path.join(VAE_DIR, "model")
VAE_TRAIN_DIR = os.path.join(VAE_DIR, "train")
VAE_TEST_DIR  = os.path.join(VAE_DIR, "test")

for d in [CNN_MODEL_DIR, CNN_TRAIN_DIR, CNN_TEST_DIR,
          IF_MODEL_DIR, IF_TRAIN_DIR, IF_TEST_DIR,
          VAE_MODEL_DIR, VAE_TRAIN_DIR, VAE_TEST_DIR]:
    os.makedirs(d, exist_ok=True)

IMG_SIZE = 200
BATCH_SIZE = 16
ACCUM_STEPS = 2
MAX_EPOCHS = 50
PATIENCE = 7

WEIGHT_DECAY = 1e-3
DROPOUT = 0.30
LABEL_SMOOTHING = 0.01

INITIAL_LR = 3e-4
BACKBONE_LR_MULTIPLIER = 0.20

FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.50

N_GOODWARE = 11000
N_MALWARE = 7000
N_TOTAL = N_GOODWARE + N_MALWARE

W_GOODWARE = N_TOTAL / (2.0 * N_GOODWARE)
W_MALWARE = N_TOTAL / (2.0 * N_MALWARE)

FROZEN_BACKBONE_LAYERS = 180

# ==============================================================
# PESI EfficientNetB0 LOCALI
# ==============================================================

EFFICIENTNET_WEIGHTS = os.path.join(BASE_DIR, "efficientnetb0_notop.h5")

if not os.path.isfile(EFFICIENTNET_WEIGHTS):
    raise FileNotFoundError(f"\nERRORE: file pesi EfficientNetB0 non trovato:\n{EFFICIENTNET_WEIGHTS}")

print(f"\nPesi EfficientNetB0 trovati: {EFFICIENTNET_WEIGHTS}")
print(f"Dimensione file: {os.path.getsize(EFFICIENTNET_WEIGHTS) / (1024**2):.2f} MB")

# ==============================================================
# GPU / MIXED PRECISION
# ==============================================================

GPUS = tf.config.list_physical_devices("GPU")
USE_AMP = len(GPUS) > 0

if USE_AMP:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")
    print(f"GPU rilevata: {GPUS[0].name}")
    print("mixed_float16: ON")
else:
    print("GPU non rilevata")
    print("mixed_float16: OFF")

with open(os.path.join(TFRECORD_DIR, "meta.json")) as f:
    meta = json.load(f)

FEATURE_COLS = meta["feature_cols"]
CSV_DIM = len(FEATURE_COLS)

print(f"Feature CSV: {CSV_DIM}")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch CNN: {BATCH_SIZE}")
print(f"Gradient accumulation CNN: {ACCUM_STEPS}")
print(f"Batch effettivo CNN: {BATCH_SIZE * ACCUM_STEPS}")
print(f"Class weight Goodware: {W_GOODWARE:.3f}")
print(f"Class weight Malware: {W_MALWARE:.3f}")

# ==============================================================
# OBFUSCATION FEATURE COLS
# ==============================================================

OBF_FEATURE_COLS = ["global_entropy", "block_entropy_mean", "block_entropy_std", "max_sect_entropy", "avg_sect_entropy", "empty_sect_count", "rwx_sections", "wx_segments", "code_data_ratio"]

# ==============================================================
# OBFUSCATION LABELER
# ==============================================================

def fit_obfuscation_labeler(df, contamination=0.15):
    scaler = StandardScaler().fit(df[OBF_FEATURE_COLS])
    X = scaler.transform(df[OBF_FEATURE_COLS])
    iso = IsolationForest(contamination=contamination, random_state=42, n_jobs=-1).fit(X)
    scores = -iso.score_samples(X)
    threshold = float(np.quantile(scores, 0.9))
    return {"scaler": scaler, "iso": iso, "threshold": threshold}

def apply_obfuscation_labels(df, labeler):
    X = labeler["scaler"].transform(df[OBF_FEATURE_COLS])
    scores = -labeler["iso"].score_samples(X)
    df = df.copy()
    df["obf_score"] = scores
    df["is_obfuscated"] = (scores > labeler["threshold"]).astype(int)
    return df

def fit_and_save_scalers():
    '''Fitta scaler CSV + obfuscation labeler sul train e li salva in CNN_MODEL_DIR
    (sono artefatti "di modello" condivisi da tutte le sezioni successive).'''
    print("\nCaricamento CSV...")
    train_df = pd.read_csv(os.path.join(BASE_DIR, "dataset_train.csv"))
    val_df = pd.read_csv(os.path.join(BASE_DIR, "dataset_val.csv"))
    test_df = pd.read_csv(os.path.join(BASE_DIR, "dataset_test.csv"))

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print("Fitting obfuscation labeler sul train...")

    obf_labeler = fit_obfuscation_labeler(train_df)

    train_df = apply_obfuscation_labels(train_df, obf_labeler)
    val_df = apply_obfuscation_labels(val_df, obf_labeler)
    test_df = apply_obfuscation_labels(test_df, obf_labeler)

    n_obf = int(train_df["is_obfuscated"].sum())
    print(f"Offuscati train: {n_obf}/{len(train_df)} ({n_obf / len(train_df) * 100:.1f}%)")

    joblib.dump(obf_labeler, os.path.join(CNN_MODEL_DIR, "obf_labeler.pkl"))

    print("Fitting StandardScaler sul train...")
    csv_scaler = StandardScaler().fit(train_df[FEATURE_COLS])
    joblib.dump(csv_scaler, os.path.join(CNN_MODEL_DIR, "csv_scaler.pkl"))

    with open(os.path.join(CNN_MODEL_DIR, "feature_cols.json"), "w") as f:
        json.dump(FEATURE_COLS, f)

    obf = {
        "train": train_df.set_index("filename")["is_obfuscated"].to_dict(),
        "val": val_df.set_index("filename")["is_obfuscated"].to_dict(),
        "test": test_df.set_index("filename")["is_obfuscated"].to_dict()
    }

    print("Scaler e obfuscation labeler salvati.")
    return obf_labeler, csv_scaler, obf

# ==============================================================
# TF.DATA
# ==============================================================

def _parse_tfrecord(proto, csv_dim):
    feat_desc = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "csv_feat": tf.io.FixedLenFeature([csv_dim], tf.float32),
        "label": tf.io.FixedLenFeature([], tf.int64),
        "filename": tf.io.FixedLenFeature([], tf.string)
    }
    parsed = tf.io.parse_single_example(proto, feat_desc)
    image = tf.io.parse_tensor(parsed["image"], out_type=tf.float32)
    image.set_shape([IMG_SIZE, IMG_SIZE, 1])
    y_mal = tf.cast(parsed["label"], tf.float32)
    return image, parsed["csv_feat"], y_mal, parsed["filename"]

def make_dataset(split_name, obf_dict, csv_scaler, shuffle, cache=True):
    tfrecord_path = os.path.join(TFRECORD_DIR, f"{split_name}.tfrecord")

    keys = list(obf_dict.keys())
    values = [obf_dict[k] for k in keys]

    table = tf.lookup.StaticHashTable(
        tf.lookup.KeyValueTensorInitializer(
            tf.constant(keys, dtype=tf.string),
            tf.constant(values, dtype=tf.int32)
        ),
        default_value=0
    )

    mean_ = tf.constant(csv_scaler.mean_, dtype=tf.float32)
    scale_ = tf.constant(csv_scaler.scale_, dtype=tf.float32)

    ds = tf.data.TFRecordDataset(tfrecord_path, num_parallel_reads=AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=BATCH_SIZE * 64, reshuffle_each_iteration=True)

    ds = ds.map(lambda proto: _parse_tfrecord(proto, CSV_DIM), num_parallel_calls=AUTOTUNE)

    def scale_and_label(image, csv_feat, y_mal, filename):
        csv_feat_scaled = (csv_feat - mean_) / (scale_ + 1e-8)
        y_obf = tf.cast(table.lookup(filename), tf.float32)
        return (image, csv_feat_scaled), (y_mal, y_obf)

    ds = ds.map(scale_and_label, num_parallel_calls=AUTOTUNE)

    if cache:
        ds = ds.cache()

    if shuffle:
        ds = ds.shuffle(buffer_size=len(obf_dict), reshuffle_each_iteration=True)

    ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    ds = ds.prefetch(AUTOTUNE)

    return ds

# ==============================================================
# CNN ARCHITETTURA (necessaria per load_trained_cnn)
# ==============================================================

def build_csv_branch(csv_in_dim, dropout=0.30):
    inp = layers.Input(shape=(csv_in_dim,), name="csv_feat")
    x = layers.Dense(256, activation="swish")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    residual = x
    x = layers.Dense(256, activation="swish")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Add()([x, residual])
    x = layers.Dense(128, activation="swish")(x)
    x = layers.BatchNormalization()(x)
    return models.Model(inp, x, name="CSVResidualMLP")

def build_malware_cnn(csv_in_dim):
    image_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="image")
    csv_in = layers.Input(shape=(csv_in_dim,), name="csv_feat")

    x_img = layers.Concatenate(name="gray_to_rgb")([image_in, image_in, image_in])

    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights=None,
        input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg", name="EfficientNetB0"
    )
    backbone.trainable = True
    x_img = backbone(x_img, training=True)

    x_img = layers.BatchNormalization(name="image_projection_bn")(x_img)
    x_img = layers.Dense(256, activation="swish", name="image_projection")(x_img)
    x_img = layers.Dropout(DROPOUT)(x_img)

    csv_branch = build_csv_branch(csv_in_dim, dropout=DROPOUT)
    x_csv = csv_branch(csv_in)
    x_csv = layers.Dense(256, activation="swish", name="csv_projection")(x_csv)
    x_csv = layers.BatchNormalization(name="csv_projection_bn")(x_csv)

    gate_input = layers.Concatenate(name="fusion_gate_input")([x_img, x_csv])
    gate = layers.Dense(256, activation="sigmoid", name="feature_gate")(gate_input)

    x_img_gated = layers.Multiply(name="image_gated")([x_img, gate])
    x_csv_gated = layers.Multiply(name="csv_gated")([x_csv, 1.0 - gate])

    shared = layers.Concatenate(name="multimodal_fusion")([x_img_gated, x_csv_gated])
    shared = layers.Dense(256, activation="swish", name="fusion_dense")(shared)
    shared = layers.BatchNormalization(name="fusion_bn")(shared)
    shared = layers.Dropout(DROPOUT)(shared)

    mal = layers.Dense(128, activation="swish", name="malware_head_dense")(shared)
    mal = layers.BatchNormalization(name="malware_head_bn")(mal)
    mal = layers.Dropout(0.20)(mal)
    out_mal = layers.Dense(1, dtype="float32", name="malware")(mal)

    obf = layers.Dense(128, activation="swish", name="obf_head_dense")(shared)
    obf = layers.BatchNormalization(name="obf_head_bn")(obf)
    obf = layers.Dropout(0.20)(obf)
    out_obf = layers.Dense(1, dtype="float32", name="obfuscated")(obf)

    model = models.Model(inputs=[image_in, csv_in], outputs=[out_mal, out_obf], name="MalwareEfficientNetB0")

    model.log_var_mal = model.add_weight(name="log_var_mal", shape=(), initializer="zeros", trainable=True, dtype=tf.float32)
    model.log_var_obf = model.add_weight(name="log_var_obf", shape=(), initializer="zeros", trainable=True, dtype=tf.float32)

    return model, backbone

def load_trained_cnn(model_path):
    '''Ricostruisce l'architettura CNN e carica solo i pesi.'''
    cnn_model, _ = build_malware_cnn(CSV_DIM)

    with zipfile.ZipFile(model_path, "r") as z:
        names = z.namelist()
        weights_candidates = [n for n in names if n.endswith(".weights.h5")]

        if not weights_candidates:
            raise FileNotFoundError(f"Nessun file pesi .weights.h5 trovato dentro {model_path}. Contenuto: {names}")

        weights_name = weights_candidates[0]
        extract_dir = "/tmp/cnn_extract"
        z.extract(weights_name, extract_dir)

    cnn_model.load_weights(os.path.join(extract_dir, weights_name))
    print(f"Pesi CNN caricati da: {weights_name}")
    return cnn_model

def extract_fusion_embeddings(cnn_model, dataset):
    '''Estrae l'embedding fuso (pre-testa) dal CNN già addestrato.'''
    embedder = models.Model(cnn_model.input, cnn_model.get_layer("fusion_bn").output)

    embs, y_mal_list, y_obf_list = [], [], []

    progress = tqdm(dataset, desc="Extract embeddings", unit="batch")

    for inputs, labels in progress:
        e = embedder(inputs, training=False)
        embs.append(tf.cast(e, tf.float32).numpy())
        y_mal_list.append(labels[0].numpy())
        y_obf_list.append(labels[1].numpy())

    return (np.concatenate(embs), np.concatenate(y_mal_list), np.concatenate(y_obf_list))

# ==============================================================
# PREDICTIONS CNN (malware + obfuscation heads)
# ==============================================================

def get_predictions(model, dataset):
    labels_mal, labels_obf, probs_mal, probs_obf = [], [], [], []

    progress = tqdm(dataset, desc="Test predictions", unit="batch")

    for inputs, (y_mal, y_obf) in progress:
        out_mal, out_obf = model(inputs, training=False)
        labels_mal.append(y_mal.numpy())
        labels_obf.append(y_obf.numpy())
        probs_mal.append(tf.sigmoid(out_mal).numpy().flatten())
        probs_obf.append(tf.sigmoid(out_obf).numpy().flatten())

    return {
        "labels_mal": np.concatenate(labels_mal).flatten(),
        "probs_mal": np.concatenate(probs_mal),
        "labels_obf": np.concatenate(labels_obf).flatten(),
        "probs_obf": np.concatenate(probs_obf)
    }

def print_classification_report(res):
    from sklearn.metrics import classification_report

    print("\n" + "=" * 70)
    print("CLASSIFICATION REPORT — MALWARE")
    print("=" * 70)
    malware_pred = (res["probs_mal"] > 0.5).astype(int)
    print(classification_report(res["labels_mal"], malware_pred, target_names=["Goodware", "Malware"]))
    try:
        print(f"ROC-AUC Malware: {roc_auc_score(res['labels_mal'], res['probs_mal']):.4f}")
    except Exception:
        pass

    print("\n" + "=" * 70)
    print("CLASSIFICATION REPORT — OFFUSCAMENTO")
    print("=" * 70)
    obf_pred = (res["probs_obf"] > 0.5).astype(int)
    print(classification_report(res["labels_obf"], obf_pred, target_names=["Non offuscato", "Offuscato"]))
    try:
        print(f"ROC-AUC Offuscamento: {roc_auc_score(res['labels_obf'], res['probs_obf']):.4f}")
    except Exception:
        pass

def plot_confusion_matrix(labels, preds, title, out_path, cmap="Blues"):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap)
    plt.title(title)
    plt.xlabel("Predetto")
    plt.ylabel("Reale")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()
    print(f"Salvato: {out_path}")

def save_plots(train_losses, val_losses, res, output_dir):
    pm = np.array(res["probs_mal"])
    lm = np.array(res["labels_mal"])
    po = np.array(res["probs_obf"])
    lo = np.array(res["labels_obf"])

    print("\nGenerazione plot CNN...")

    if len(train_losses) > 0 and len(val_losses) > 0:
        plt.figure(figsize=(8, 4))
        plt.plot(train_losses, label="Train")
        plt.plot(val_losses, label="Val")
        plt.title("Loss per Epoca")
        plt.xlabel("Epoca")
        plt.ylabel("Loss")
        plt.legend()
        plt.tight_layout()
        p = os.path.join(output_dir, "loss_curve.png")
        plt.savefig(p)
        plt.close()
        print(f"Salvato: {p}")

    plot_confusion_matrix(lm, (pm > 0.5).astype(int), "Confusion Matrix - Malware",
                           os.path.join(output_dir, "confusion_matrix_malware.png"))

    plot_confusion_matrix(lo, (po > 0.5).astype(int), "Confusion Matrix - Offuscato",
                           os.path.join(output_dir, "confusion_matrix_obfuscation.png"), cmap="Oranges")

    df_res = pd.DataFrame({"Confidenza": pm, "Classe": lm})
    df_res["Classe"] = df_res["Classe"].map({0: "Goodware", 1: "Malware"})

    plt.figure(figsize=(8, 6))
    sns.boxplot(x="Classe", y="Confidenza", data=df_res, hue="Classe",
                palette={"Goodware": "blue", "Malware": "red"}, legend=False)
    plt.axhline(0.5, color="black", linestyle="--")
    plt.tight_layout()
    p = os.path.join(output_dir, "confidence_boxplot_malware.png")
    plt.savefig(p)
    plt.close()
    print(f"Salvato: {p}")

    for name, labels, probs, fname in [("Malware", lm, pm, "malware"), ("Offuscamento", lo, po, "obfuscation")]:
        fpr, tpr, _ = roc_curve(labels, probs)
        roc_auc = auc(fpr, tpr)

        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, label=f"AUC={roc_auc:.4f}")
        plt.plot([0, 1], [0, 1], "k--")
        plt.title(f"ROC - {name}")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.legend()
        plt.tight_layout()
        p = os.path.join(output_dir, f"roc_curve_{fname}.png")
        plt.savefig(p)
        plt.close()
        print(f"ROC {name}: AUC={roc_auc:.4f}")

        prec, rec, _ = precision_recall_curve(labels, probs)
        plt.figure(figsize=(6, 5))
        plt.plot(rec, prec)
        plt.title(f"PR Curve - {name}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.tight_layout()
        p = os.path.join(output_dir, f"pr_curve_{fname}.png")
        plt.savefig(p)
        plt.close()
        print(f"PR salvata: {p}")

# ==============================================================
# CONVERSIONE ERRORE/SCORE → PERCENTUALE (comune a IF e VAE)
# ==============================================================

def errors_to_percentage(errors, err_min, err_max):
    return np.clip((errors - err_min) / (err_max - err_min + 1e-8), 0, 1) * 100

print("\nBase condivisa caricata. Procedi con la sezione 3 (CNN), 4 (Isolation Forest) o 5 (VAE).")

---
## 3. Modello CNN — EfficientNetB0 multitask

CNN principale: EfficientNetB0 sull'immagine + MLP residuale sulle feature CSV, fuse con un meccanismo di gating, con due teste d'uscita: **malware** e **offuscamento**. Loss: focal loss pesata per il malware + BCE per l'offuscamento, combinate con uncertainty weighting omoschedastico appreso durante il training.

In [ ]:
# ==============================================================
# SEZIONE 3a — CNN: FOCAL LOSS, TRAINER, TRAINING LOOP
# Richiede la sezione 2 (BASE CONDIVISA) già eseguita.
# ==============================================================

# ==============================================================
# FOCAL LOSS
# ==============================================================

def focal_binary_loss(y_true, logits, gamma=2.0, alpha=0.5, label_smoothing=0.0):
    y_true = tf.cast(y_true, tf.float32)

    if label_smoothing > 0:
        y_true = y_true * (1.0 - label_smoothing) + 0.5 * label_smoothing

    probs = tf.sigmoid(logits)
    probs = tf.clip_by_value(probs, 1e-7, 1.0 - 1e-7)

    bce = -(y_true * tf.math.log(probs) + (1.0 - y_true) * tf.math.log(1.0 - probs))
    p_t = y_true * probs + (1.0 - y_true) * (1.0 - probs)
    alpha_t = y_true * alpha + (1.0 - y_true) * (1.0 - alpha)
    focal = alpha_t * tf.pow(1.0 - p_t, gamma) * bce

    return focal

def weighted_focal_loss(y_true, logits):
    base = focal_binary_loss(y_true, logits, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA, label_smoothing=LABEL_SMOOTHING)
    original_y = tf.cast(y_true, tf.float32)
    weights = tf.where(tf.equal(original_y, 1.0), W_MALWARE, W_GOODWARE)
    return base * weights

# ==============================================================
# MULTITASK UNCERTAINTY LOSS
# ==============================================================

def uncertainty_multitask_loss(model, malware_loss, obf_loss):
    precision_mal = tf.exp(-model.log_var_mal)
    precision_obf = tf.exp(-model.log_var_obf)

    total = precision_mal * malware_loss + model.log_var_mal
    total += precision_obf * obf_loss + model.log_var_obf

    return total

# ==============================================================
# TRAINER CNN — gradient accumulation
# ==============================================================

class AdvancedAccumulatingTrainer:
    def __init__(self, model, optimizer):
        self.model = model
        self.optimizer = optimizer
        self.accum_steps = ACCUM_STEPS
        self.trainable_vars = list(model.trainable_variables)

    def _zero_grads(self):
        return [tf.zeros_like(v, dtype=tf.float32) for v in self.trainable_vars]

    @tf.function(jit_compile=False)
    def _train_step(self, inputs, labels, accum_grads):
        with tf.GradientTape() as tape:
            preds = self.model(inputs, training=True)

            y_mal = tf.expand_dims(labels[0], -1)
            y_obf = tf.expand_dims(labels[1], -1)

            mal_loss = tf.reduce_mean(weighted_focal_loss(y_mal, preds[0]))
            obf_loss_raw = tf.keras.losses.binary_crossentropy(y_obf, preds[1], from_logits=True)
            obf_loss = tf.reduce_mean(obf_loss_raw)

            total_loss = uncertainty_multitask_loss(self.model, mal_loss, obf_loss)
            total_loss = total_loss / tf.cast(self.accum_steps, tf.float32)

        grads = tape.gradient(total_loss, self.trainable_vars)

        new_accum = [
            a + tf.cast(g, tf.float32) if g is not None else a
            for a, g in zip(accum_grads, grads)
        ]

        return total_loss, mal_loss, obf_loss, new_accum

    @tf.function(jit_compile=False)
    def _apply_grads(self, accum_grads):
        self.optimizer.apply_gradients(zip(accum_grads, self.trainable_vars))

    @tf.function(jit_compile=False)
    def _val_step(self, inputs, labels):
        preds = self.model(inputs, training=False)

        y_mal = tf.expand_dims(labels[0], -1)
        y_obf = tf.expand_dims(labels[1], -1)

        mal_loss = tf.reduce_mean(weighted_focal_loss(y_mal, preds[0]))
        obf_loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_obf, preds[1], from_logits=True))

        total_loss = uncertainty_multitask_loss(self.model, mal_loss, obf_loss)

        return total_loss, mal_loss, obf_loss

    def train_epoch(self, dataset):
        accum_grads = self._zero_grads()
        total_loss = total_mal = total_obf = 0.0
        step = 0

        progress = tqdm(dataset, desc="Training", unit="batch", leave=False)

        for inputs, labels in progress:
            loss, mal_loss, obf_loss, accum_grads = self._train_step(inputs, labels, accum_grads)

            total_loss += float(loss) * self.accum_steps
            total_mal += float(mal_loss)
            total_obf += float(obf_loss)
            step += 1

            if step % self.accum_steps == 0:
                self._apply_grads(accum_grads)
                accum_grads = self._zero_grads()

            progress.set_postfix(loss=f"{float(loss):.4f}", mal=f"{float(mal_loss):.4f}", obf=f"{float(obf_loss):.4f}")

        if step % self.accum_steps != 0:
            self._apply_grads(accum_grads)

        return total_loss / max(step, 1), total_mal / max(step, 1), total_obf / max(step, 1)

    def val_epoch(self, dataset):
        total = total_mal = total_obf = 0.0
        n = 0

        progress = tqdm(dataset, desc="Validation", unit="batch", leave=False)

        for inputs, labels in progress:
            loss, mal_loss, obf_loss = self._val_step(inputs, labels)
            total += float(loss)
            total_mal += float(mal_loss)
            total_obf += float(obf_loss)
            n += 1
            progress.set_postfix(loss=f"{float(loss):.4f}")

        return total / max(n, 1), total_mal / max(n, 1), total_obf / max(n, 1)

def _get_current_lr(optimizer):
    lr = optimizer.learning_rate
    if callable(lr):
        return float(lr(optimizer.iterations))
    return float(lr)

# ==============================================================
# CNN TRAINING
# ==============================================================

def train_cnn(train_ds, val_ds):
    model, backbone = build_malware_cnn(CSV_DIM)

    backbone.trainable = True
    for layer in backbone.layers[:FROZEN_BACKBONE_LAYERS]:
        layer.trainable = False

    n_trainable_backbone = sum(1 for l in backbone.layers if l.trainable)
    print(f"\nBackbone: {len(backbone.layers)} layer totali | {n_trainable_backbone} trainable | {FROZEN_BACKBONE_LAYERS} congelati")

    train_steps = int(np.ceil((N_TOTAL * 0.70) / BATCH_SIZE))
    decay_steps = MAX_EPOCHS * train_steps

    lr_schedule = optimizers.schedules.CosineDecay(initial_learning_rate=INITIAL_LR, decay_steps=decay_steps, alpha=1e-5)

    optimizer = optimizers.AdamW(learning_rate=lr_schedule, weight_decay=WEIGHT_DECAY, beta_1=0.9, beta_2=0.999, clipnorm=1.0)

    trainer = AdvancedAccumulatingTrainer(model, optimizer)

    best_val = float("inf")
    patience_cnt = 0
    best_weights = None
    train_losses, val_losses = [], []

    best_model_path = os.path.join(CNN_MODEL_DIR, "best_model.keras")

    print("\n" + "=" * 70)
    print("CNN TRAINING — EfficientNetB0 + CSV RESIDUAL MLP")
    print("=" * 70)
    print(f"Batch: {BATCH_SIZE} | Accum: {ACCUM_STEPS} | Effective batch: {BATCH_SIZE * ACCUM_STEPS}")
    print(f"Image: {IMG_SIZE}x{IMG_SIZE}")
    print(f"Parameters: {model.count_params():,}")
    print(f"Initial LR: {INITIAL_LR:.2e}")
    print("Backbone: EfficientNetB0 con pesi locali")
    print(f"Weights: {EFFICIENTNET_WEIGHTS}")
    print("Fusion: gated multimodal fusion")
    print("Loss: focal malware + BCE obfuscation + uncertainty weighting")
    print("=" * 70)

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()

        t_loss, t_mal, t_obf = trainer.train_epoch(train_ds)
        v_loss, v_mal, v_obf = trainer.val_epoch(val_ds)

        elapsed = time.time() - t0
        lr_now = _get_current_lr(optimizer)

        train_losses.append(t_loss)
        val_losses.append(v_loss)

        sigma_mal = float(tf.exp(0.5 * model.log_var_mal))
        sigma_obf = float(tf.exp(0.5 * model.log_var_obf))

        if v_loss < best_val:
            best_val = v_loss
            patience_cnt = 0
            best_weights = model.get_weights()
            model.save(best_model_path)
            marker = "✓ BEST"
        else:
            patience_cnt += 1
            marker = ""

        print(
            f"Epoch {epoch:02d}/{MAX_EPOCHS} | train={t_loss:.4f} | val={v_loss:.4f} | "
            f"mal={v_mal:.4f} | obf={v_obf:.4f} | lr={lr_now:.2e} | "
            f"sigma_mal={sigma_mal:.3f} | sigma_obf={sigma_obf:.3f} | "
            f"time={elapsed:.0f}s | patience={patience_cnt}/{PATIENCE} {marker}"
        )

        if patience_cnt >= PATIENCE:
            print(f"\nEarly stopping a epoch {epoch}.")
            break

    if best_weights is not None:
        model.set_weights(best_weights)

    print(f"\nBest validation loss: {best_val:.4f}")
    print(f"Learned malware uncertainty: {float(model.log_var_mal):.4f}")
    print(f"Learned obfuscation uncertainty: {float(model.log_var_obf):.4f}")

    return model, train_losses, val_losses

print("Funzioni di training CNN definite. Esegui la cella successiva per avviare l'addestramento.")

**Avvio training CNN.** Fitta scaler/labeler, costruisce train/val dataset e addestra il CNN da zero, salvando il modello migliore in `CNN_MODEL_DIR` e i grafici (loss curve, confusion matrix, ROC, PR) in `CNN_TRAIN_DIR`.

In [ ]:
# ==============================================================
# SEZIONE 3b — CNN: MAIN TRAIN
# ==============================================================

def main_train_cnn():
    t_start = time.time()

    print("\n" + "=" * 70)
    print("1/3 — FITTING SCALER + OBFUSCATION LABELER")
    print("=" * 70)
    _, csv_scaler, obf_dicts = fit_and_save_scalers()

    print("\n" + "=" * 70)
    print("2/3 — COSTRUZIONE DATASET TF.DATA")
    print("=" * 70)
    print("Creazione train dataset...")
    train_ds = make_dataset("train", obf_dicts["train"], csv_scaler, shuffle=True, cache=True)
    print("Creazione validation dataset...")
    val_ds = make_dataset("val", obf_dicts["val"], csv_scaler, shuffle=False, cache=True)

    print("\n" + "=" * 70)
    print("3/3 — TRAINING CNN")
    print("=" * 70)
    model, train_losses, val_losses = train_cnn(train_ds, val_ds)

    print("\nValutazione su validation set...")
    res = get_predictions(model, val_ds)
    print_classification_report(res)
    save_plots(train_losses, val_losses, res, output_dir=CNN_TRAIN_DIR)

    elapsed_total = (time.time() - t_start) / 60
    print("\n" + "=" * 70)
    print(f"TRAINING CNN COMPLETATO in {elapsed_total:.1f} minuti")
    print(f"Modello: {CNN_MODEL_DIR}")
    print(f"Plot training: {CNN_TRAIN_DIR}")
    print("=" * 70)

    return model

cnn_model = main_train_cnn()

**Test CNN.** Richiede il CNN già addestrato (`best_model.keras` in `CNN_MODEL_DIR`). Carica il modello, valuta sul test set e salva classification report + grafici in `CNN_TEST_DIR`.

In [ ]:
# ==============================================================
# SEZIONE 3c — CNN: MAIN TEST
# ==============================================================

def main_test_cnn():
    t_start = time.time()

    print("\n" + "=" * 70)
    print("TEST — CNN")
    print("=" * 70)

    csv_scaler_path = os.path.join(CNN_MODEL_DIR, "csv_scaler.pkl")
    obf_labeler_path = os.path.join(CNN_MODEL_DIR, "obf_labeler.pkl")

    if not os.path.isfile(csv_scaler_path) or not os.path.isfile(obf_labeler_path):
        raise FileNotFoundError("\nScaler o obfuscation labeler non trovati. Esegui prima main_train_cnn().")

    csv_scaler = joblib.load(csv_scaler_path)
    obf_labeler = joblib.load(obf_labeler_path)

    print("Caricamento e labeling test set...")
    test_df = pd.read_csv(os.path.join(BASE_DIR, "dataset_test.csv"))
    test_df = apply_obfuscation_labels(test_df, obf_labeler)
    obf_test = test_df.set_index("filename")["is_obfuscated"].to_dict()

    print("Creazione test dataset...")
    test_ds = make_dataset("test", obf_test, csv_scaler, shuffle=False, cache=True)

    best_model_path = os.path.join(CNN_MODEL_DIR, "best_model.keras")
    cnn_model = load_trained_cnn(best_model_path)

    res = get_predictions(cnn_model, test_ds)
    print_classification_report(res)
    save_plots([], [], res, output_dir=CNN_TEST_DIR)

    elapsed_total = (time.time() - t_start) / 60
    print("\n" + "=" * 70)
    print(f"TEST CNN COMPLETATO in {elapsed_total:.1f} minuti")
    print(f"Output: {CNN_TEST_DIR}")
    print("=" * 70)

    return cnn_model

cnn_model = main_test_cnn()

---
## 4. Modello Isolation Forest su embedding

Approccio non supervisionato all'offuscamento, alternativo al VAE: estrae l'embedding fuso dal CNN già addestrato (sezione 3) e addestra un Isolation Forest sugli embedding del train. L'anomaly score calibrato (min-max sul train) viene poi confrontato con l'etichetta `is_obfuscated` sia in fase di training (per verifica) sia in fase di test.

Modello e calibrazione salvati in `IF_MODEL_DIR`; plot di calibrazione sul train in `IF_TRAIN_DIR`; plot e CSV di test in `IF_TEST_DIR`.

In [ ]:
# ==============================================================
# SEZIONE 4a — ISOLATION FOREST: TRAINING SU EMBEDDING
# Richiede: sezione 2 (BASE CONDIVISA) + CNN già addestrato (sezione 3).
# ==============================================================

IF_N_ESTIMATORS = 300
IF_CONTAMINATION = 0.15

def plot_if_results(scores, y_obf, pct, output_dir):
    print("\nGenerazione plot Isolation Forest...")

    plt.figure(figsize=(8, 5))
    sns.histplot(scores[y_obf == 0], color="blue", label="Non offuscato", kde=True, stat="density", alpha=0.4)
    sns.histplot(scores[y_obf == 1], color="red", label="Offuscato", kde=True, stat="density", alpha=0.4)
    plt.title("Distribuzione Anomaly Score — Isolation Forest")
    plt.xlabel("Anomaly Score")
    plt.ylabel("Densità")
    plt.legend()
    plt.tight_layout()
    p = os.path.join(output_dir, "if_score_distribution.png")
    plt.savefig(p)
    plt.close()
    print(f"Salvato: {p}")

    df_score = pd.DataFrame({"Score": scores, "Classe": y_obf})
    df_score["Classe"] = df_score["Classe"].map({0: "Non offuscato", 1: "Offuscato"})

    plt.figure(figsize=(7, 5))
    sns.boxplot(x="Classe", y="Score", data=df_score, hue="Classe",
                palette={"Non offuscato": "blue", "Offuscato": "red"}, legend=False)
    plt.title("Anomaly Score per Classe")
    plt.tight_layout()
    p = os.path.join(output_dir, "if_score_boxplot.png")
    plt.savefig(p)
    plt.close()
    print(f"Salvato: {p}")

    fpr, tpr, _ = roc_curve(y_obf, scores)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], "k--")
    plt.title("ROC — Isolation Forest (obfuscation)")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.tight_layout()
    p = os.path.join(output_dir, "if_roc_curve.png")
    plt.savefig(p)
    plt.close()
    print(f"ROC IF: AUC={roc_auc:.4f}")

    prec, rec, _ = precision_recall_curve(y_obf, scores)
    plt.figure(figsize=(6, 5))
    plt.plot(rec, prec)
    plt.title("PR Curve — Isolation Forest (obfuscation)")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.tight_layout()
    p = os.path.join(output_dir, "if_pr_curve.png")
    plt.savefig(p)
    plt.close()
    print(f"Salvato: {p}")

    plt.figure(figsize=(7, 5))
    plt.scatter(scores, pct, c=y_obf, cmap="coolwarm", alpha=0.5, s=10)
    plt.title("Score vs Percentuale Calibrata")
    plt.xlabel("Anomaly Score")
    plt.ylabel("Obf % (calibrata)")
    plt.colorbar(label="is_obfuscated")
    plt.tight_layout()
    p = os.path.join(output_dir, "if_score_vs_pct.png")
    plt.savefig(p)
    plt.close()
    print(f"Salvato: {p}")

def train_embedding_isolation_forest(train_emb):
    print("\n" + "=" * 70)
    print("ISOLATION FOREST TRAINING — EMBEDDING (alternativa VAE)")
    print("=" * 70)
    print(f"Input dim: {train_emb.shape[1]}")
    print(f"N estimators: {IF_N_ESTIMATORS}")
    print(f"Contamination: {IF_CONTAMINATION}")
    print(f"Train samples: {len(train_emb)}")
    print("=" * 70)

    t0 = time.time()

    iso = IsolationForest(
        n_estimators=IF_N_ESTIMATORS,
        contamination=IF_CONTAMINATION,
        random_state=42,
        n_jobs=-1
    ).fit(train_emb)

    elapsed = time.time() - t0
    print(f"Fit completato in {elapsed:.1f}s")

    return iso

def compute_if_anomaly_scores(iso, embeddings):
    '''Score più alto = più anomalo (coerente con reconstruction error).'''
    return -iso.score_samples(embeddings)

def main_train_if():
    t_start = time.time()

    print("\n" + "=" * 70)
    print("1/3 — FITTING SCALER + OBFUSCATION LABELER")
    print("=" * 70)
    _, csv_scaler, obf_dicts = fit_and_save_scalers()

    print("\n" + "=" * 70)
    print("2/3 — COSTRUZIONE DATASET TF.DATA")
    print("=" * 70)
    print("Creazione train dataset...")
    train_ds = make_dataset("train", obf_dicts["train"], csv_scaler, shuffle=True, cache=True)

    print("\n" + "=" * 70)
    print("3/3 — TRAINING ISOLATION FOREST SU EMBEDDING")
    print("=" * 70)

    best_model_path = os.path.join(CNN_MODEL_DIR, "best_model.keras")
    if not os.path.isfile(best_model_path):
        raise FileNotFoundError(f"\nCNN non trovata in: {best_model_path}\nAddestra prima il CNN (sezione 3) prima di lanciare main_train_if().")

    print("Caricamento CNN già addestrata...")
    cnn_model = load_trained_cnn(best_model_path)

    print("Estrazione embedding train...")
    train_emb, _, y_obf_train = extract_fusion_embeddings(cnn_model, train_ds)

    iso_model = train_embedding_isolation_forest(train_emb)
    joblib.dump(iso_model, os.path.join(IF_MODEL_DIR, "iso_embedding.pkl"))

    print("\nCalibrazione anomaly score sul train...")
    train_scores = compute_if_anomaly_scores(iso_model, train_emb)
    score_min = float(train_scores.min())
    score_max = float(train_scores.max())
    print(f"Train anomaly score → min={score_min:.6f} | max={score_max:.6f}")

    calibration_path = os.path.join(IF_MODEL_DIR, "if_calibration.json")
    with open(calibration_path, "w") as f:
        json.dump({
            "score_min": score_min,
            "score_max": score_max,
            "embedding_source": "fusion_bn",
            "n_estimators": IF_N_ESTIMATORS,
            "contamination": IF_CONTAMINATION
        }, f)
    print(f"Calibrazione salvata: {calibration_path}")

    print("\nGenerazione plot di calibrazione (train)...")
    train_pct = errors_to_percentage(train_scores, score_min, score_max)
    plot_if_results(train_scores, y_obf_train, train_pct, IF_TRAIN_DIR)

    try:
        if_auc_train = roc_auc_score(y_obf_train, train_scores)
        print(f"IF ROC-AUC (train, anomaly score vs is_obfuscated): {if_auc_train:.4f}")
    except Exception as e:
        print(f"Impossibile calcolare IF ROC-AUC train: {e}")

    elapsed_total = (time.time() - t_start) / 60
    print("\n" + "=" * 70)
    print(f"TRAINING ISOLATION FOREST COMPLETATO in {elapsed_total:.1f} minuti")
    print(f"Modello: {IF_MODEL_DIR}")
    print(f"Plot training: {IF_TRAIN_DIR}")
    print("=" * 70)

    return iso_model

iso_model = main_train_if()

**Test Isolation Forest.** Richiede CNN e Isolation Forest già addestrati. Valuta sul test set, produce grafici specifici IF (distribuzione score, boxplot, ROC, PR, scatter) in `IF_TEST_DIR` e un CSV con la percentuale di offuscamento stimata per ogni sample di test.

In [ ]:
# ==============================================================
# SEZIONE 4b — ISOLATION FOREST: TEST + PLOT
# ==============================================================

def main_test_if():
    t_start = time.time()

    print("\n" + "=" * 70)
    print("TEST — CNN + ISOLATION FOREST (embedding)")
    print("=" * 70)

    csv_scaler_path = os.path.join(CNN_MODEL_DIR, "csv_scaler.pkl")
    obf_labeler_path = os.path.join(CNN_MODEL_DIR, "obf_labeler.pkl")

    csv_scaler = joblib.load(csv_scaler_path)
    obf_labeler = joblib.load(obf_labeler_path)

    print("Caricamento e labeling test set...")
    test_df = pd.read_csv(os.path.join(BASE_DIR, "dataset_test.csv"))
    test_df = apply_obfuscation_labels(test_df, obf_labeler)
    obf_test = test_df.set_index("filename")["is_obfuscated"].to_dict()

    test_ds = make_dataset("test", obf_test, csv_scaler, shuffle=False, cache=True)

    print("\n" + "-" * 70)
    print("CNN — VALUTAZIONE")
    print("-" * 70)

    best_model_path = os.path.join(CNN_MODEL_DIR, "best_model.keras")
    cnn_model = load_trained_cnn(best_model_path)

    res = get_predictions(cnn_model, test_ds)
    print_classification_report(res)
    save_plots([], [], res, output_dir=CNN_TEST_DIR)

    print("\n" + "-" * 70)
    print("ISOLATION FOREST — VALUTAZIONE")
    print("-" * 70)

    iso_model_path = os.path.join(IF_MODEL_DIR, "iso_embedding.pkl")
    calib_path = os.path.join(IF_MODEL_DIR, "if_calibration.json")

    iso_model = joblib.load(iso_model_path)
    with open(calib_path) as f:
        calib = json.load(f)

    print("Estrazione embedding test...")
    test_emb, _, y_obf_test = extract_fusion_embeddings(cnn_model, test_ds)

    print("Calcolo anomaly score...")
    test_scores = compute_if_anomaly_scores(iso_model, test_emb)
    test_pct = errors_to_percentage(test_scores, calib["score_min"], calib["score_max"])

    try:
        if_auc = roc_auc_score(y_obf_test, test_scores)
        print(f"IF ROC-AUC (anomaly score vs is_obfuscated): {if_auc:.4f}")
    except Exception as e:
        print(f"Impossibile calcolare IF ROC-AUC: {e}")

    plot_if_results(test_scores, y_obf_test, test_pct, IF_TEST_DIR)

    print("Lettura filename del test TFRecord...")
    filenames = []
    test_path = os.path.join(TFRECORD_DIR, "test.tfrecord")

    for proto in tf.data.TFRecordDataset(test_path):
        p = tf.io.parse_single_example(proto, {"filename": tf.io.FixedLenFeature([], tf.string)})
        filenames.append(p["filename"].numpy().decode())

    n = min(len(filenames), len(test_pct))

    out_df = pd.DataFrame({
        "filename": filenames[:n],
        "obf_pct_unsupervised": test_pct[:n]
    })

    csv_out = os.path.join(IF_TEST_DIR, "obfuscation_percentage_test_if.csv")
    out_df.to_csv(csv_out, index=False)

    elapsed_total = (time.time() - t_start) / 60
    print("\n" + "=" * 70)
    print(f"TEST ISOLATION FOREST COMPLETATO in {elapsed_total:.1f} minuti")
    print(f"Output: {IF_TEST_DIR}")
    print(f"CSV obfuscation: {csv_out}")
    print("=" * 70)

main_test_if()

---
## 5. Modello VAE su embedding

Alternativa neurale all'Isolation Forest: un Variational Autoencoder addestrato sugli stessi embedding fusi del CNN. Il reconstruction error (MSE) funge da anomaly score, calibrato min-max sul train. Il VAE è forzato in float32 indipendentemente dalla policy di mixed precision globale, per stabilità numerica.

Encoder/decoder e calibrazione salvati in `VAE_MODEL_DIR`; plot di training/validation in `VAE_TRAIN_DIR`; plot e CSV di test in `VAE_TEST_DIR`.

In [ ]:
# ==============================================================
# SEZIONE 5a — VAE: ARCHITETTURA E TRAINING SU EMBEDDING
# Richiede: sezione 2 (BASE CONDIVISA) + CNN già addestrato (sezione 3).
# Fix mixed_precision: VAE forzato in float32.
# ==============================================================

VAE_LATENT_DIM = 32
VAE_EPOCHS = 100
VAE_PATIENCE = 10
VAE_BATCH_SIZE = 64
VAE_LR = 1e-3
KL_WEIGHT = 0.001


class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = tf.random.normal(shape=tf.shape(z_mean), dtype=z_mean.dtype)
        return z_mean + tf.exp(0.5 * z_log_var) * eps


def build_vae(input_dim, latent_dim=VAE_LATENT_DIM):
    # Encoder — forzato float32 indipendentemente dalla policy globale mixed_float16
    enc_in = layers.Input(shape=(input_dim,), name="vae_input", dtype="float32")
    x = layers.Dense(128, activation="swish", dtype="float32")(enc_in)
    x = layers.BatchNormalization(dtype="float32")(x)
    x = layers.Dense(64, activation="swish", dtype="float32")(x)
    x = layers.BatchNormalization(dtype="float32")(x)

    z_mean = layers.Dense(latent_dim, name="z_mean", dtype="float32")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var", dtype="float32")(x)
    z = Sampling(name="z", dtype="float32")([z_mean, z_log_var])

    encoder = models.Model(enc_in, [z_mean, z_log_var, z], name="VAE_Encoder")

    # Decoder
    dec_in = layers.Input(shape=(latent_dim,), name="latent_input", dtype="float32")
    y = layers.Dense(64, activation="swish", dtype="float32")(dec_in)
    y = layers.BatchNormalization(dtype="float32")(y)
    y = layers.Dense(128, activation="swish", dtype="float32")(y)
    y = layers.BatchNormalization(dtype="float32")(y)
    out = layers.Dense(input_dim, activation=None, name="reconstruction", dtype="float32")(y)

    decoder = models.Model(dec_in, out, name="VAE_Decoder")

    return encoder, decoder


class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.recon_loss_tracker = tf.keras.metrics.Mean(name="recon_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.recon_loss_tracker, self.kl_loss_tracker]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data, training=True)
            recon = self.decoder(z, training=True)

            recon_loss = tf.reduce_mean(tf.reduce_sum(tf.square(data - recon), axis=1))
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
            )
            total_loss = recon_loss + KL_WEIGHT * kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        z_mean, z_log_var, z = self.encoder(data, training=False)
        recon = self.decoder(z, training=False)

        recon_loss = tf.reduce_mean(tf.reduce_sum(tf.square(data - recon), axis=1))
        kl_loss = -0.5 * tf.reduce_mean(
            tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
        )
        total_loss = recon_loss + KL_WEIGHT * kl_loss

        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {m.name: m.result() for m in self.metrics}

    def call(self, data):
        z_mean, z_log_var, z = self.encoder(data)
        return self.decoder(z)


def compute_vae_reconstruction_errors(vae, embeddings, batch_size=512):
    '''Reconstruction error per-sample (MSE), score alto = più anomalo.'''
    errors = []
    embeddings = embeddings.astype(np.float32)
    for i in range(0, len(embeddings), batch_size):
        batch = embeddings[i:i + batch_size]
        z_mean, z_log_var, z = vae.encoder(batch, training=False)
        recon = vae.decoder(z_mean, training=False)  # z_mean per determinismo in inferenza
        err = np.mean(np.square(batch - recon.numpy()), axis=1)
        errors.append(err)
    return np.concatenate(errors)


def train_vae_on_embeddings(train_emb, val_emb):
    print("\n" + "=" * 70)
    print("VAE TRAINING — EMBEDDING (alternativa Isolation Forest)")
    print("=" * 70)
    print(f"Input dim: {train_emb.shape[1]}")
    print(f"Latent dim: {VAE_LATENT_DIM}")
    print(f"Batch size: {VAE_BATCH_SIZE}")
    print(f"Max epochs: {VAE_EPOCHS} | Patience: {VAE_PATIENCE}")
    print("=" * 70)

    train_emb = train_emb.astype(np.float32)
    val_emb = val_emb.astype(np.float32)

    encoder, decoder = build_vae(train_emb.shape[1], VAE_LATENT_DIM)
    vae = VAE(encoder, decoder)
    vae.compile(optimizer=optimizers.Adam(learning_rate=VAE_LR))

    train_ds = tf.data.Dataset.from_tensor_slices(train_emb)
    train_ds = train_ds.shuffle(len(train_emb)).batch(VAE_BATCH_SIZE).prefetch(AUTOTUNE)

    val_ds = tf.data.Dataset.from_tensor_slices(val_emb)
    val_ds = val_ds.batch(VAE_BATCH_SIZE).prefetch(AUTOTUNE)

    best_val = float("inf")
    patience_cnt = 0
    best_weights = None
    train_losses, val_losses = [], []

    for epoch in range(1, VAE_EPOCHS + 1):
        t0 = time.time()

        for m in vae.metrics:
            m.reset_state()
        for batch in train_ds:
            logs = vae.train_step(batch)
        t_loss = float(logs["loss"])

        for m in vae.metrics:
            m.reset_state()
        for batch in val_ds:
            logs = vae.test_step(batch)
        v_loss = float(logs["loss"])

        train_losses.append(t_loss)
        val_losses.append(v_loss)
        elapsed = time.time() - t0

        if v_loss < best_val:
            best_val = v_loss
            patience_cnt = 0
            best_weights = vae.get_weights()
            marker = "✓ BEST"
        else:
            patience_cnt += 1
            marker = ""

        print(f"Epoch {epoch:03d}/{VAE_EPOCHS} | train={t_loss:.4f} | val={v_loss:.4f} | "
              f"time={elapsed:.1f}s | patience={patience_cnt}/{VAE_PATIENCE} {marker}")

        if patience_cnt >= VAE_PATIENCE:
            print(f"\nEarly stopping a epoch {epoch}.")
            break

    if best_weights is not None:
        vae.set_weights(best_weights)

    print(f"\nBest validation loss: {best_val:.4f}")
    return vae, train_losses, val_losses


def plot_vae_results(train_losses, val_losses, errors, y_obf, pct, output_dir):
    print("\nGenerazione plot VAE...")

    if len(train_losses) > 0 and len(val_losses) > 0:
        plt.figure(figsize=(8, 4))
        plt.plot(train_losses, label="Train")
        plt.plot(val_losses, label="Val")
        plt.title("VAE — Loss per Epoca")
        plt.xlabel("Epoca")
        plt.ylabel("Loss (recon + KL)")
        plt.legend()
        plt.tight_layout()
        p = os.path.join(output_dir, "vae_loss_curve.png")
        plt.savefig(p)
        plt.close()
        print(f"Salvato: {p}")

    plt.figure(figsize=(8, 5))
    sns.histplot(errors[y_obf == 0], color="blue", label="Non offuscato", kde=True, stat="density", alpha=0.4)
    sns.histplot(errors[y_obf == 1], color="red", label="Offuscato", kde=True, stat="density", alpha=0.4)
    plt.title("Distribuzione Reconstruction Error — VAE")
    plt.xlabel("Reconstruction Error (MSE)")
    plt.ylabel("Densità")
    plt.legend()
    plt.tight_layout()
    p = os.path.join(output_dir, "vae_error_distribution.png")
    plt.savefig(p)
    plt.close()
    print(f"Salvato: {p}")

    df_err = pd.DataFrame({"Errore": errors, "Classe": y_obf})
    df_err["Classe"] = df_err["Classe"].map({0: "Non offuscato", 1: "Offuscato"})

    plt.figure(figsize=(7, 5))
    sns.boxplot(x="Classe", y="Errore", data=df_err, hue="Classe",
                palette={"Non offuscato": "blue", "Offuscato": "red"}, legend=False)
    plt.title("Reconstruction Error per Classe")
    plt.tight_layout()
    p = os.path.join(output_dir, "vae_error_boxplot.png")
    plt.savefig(p)
    plt.close()
    print(f"Salvato: {p}")

    fpr, tpr, _ = roc_curve(y_obf, errors)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], "k--")
    plt.title("ROC — VAE (obfuscation)")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.tight_layout()
    p = os.path.join(output_dir, "vae_roc_curve.png")
    plt.savefig(p)
    plt.close()
    print(f"ROC VAE: AUC={roc_auc:.4f}")

    prec, rec, _ = precision_recall_curve(y_obf, errors)
    plt.figure(figsize=(6, 5))
    plt.plot(rec, prec)
    plt.title("PR Curve — VAE (obfuscation)")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.tight_layout()
    p = os.path.join(output_dir, "vae_pr_curve.png")
    plt.savefig(p)
    plt.close()
    print(f"Salvato: {p}")

    plt.figure(figsize=(7, 5))
    plt.scatter(errors, pct, c=y_obf, cmap="coolwarm", alpha=0.5, s=10)
    plt.title("Reconstruction Error vs Percentuale Calibrata")
    plt.xlabel("Reconstruction Error")
    plt.ylabel("Obf % (calibrata)")
    plt.colorbar(label="is_obfuscated")
    plt.tight_layout()
    p = os.path.join(output_dir, "vae_error_vs_pct.png")
    plt.savefig(p)
    plt.close()
    print(f"Salvato: {p}")


def main_train_vae():
    t_start = time.time()

    print("\n" + "=" * 70)
    print("CARICAMENTO SCALER + LABELER + DATASET")
    print("=" * 70)

    csv_scaler = joblib.load(os.path.join(CNN_MODEL_DIR, "csv_scaler.pkl"))
    obf_labeler = joblib.load(os.path.join(CNN_MODEL_DIR, "obf_labeler.pkl"))

    train_df = pd.read_csv(os.path.join(BASE_DIR, "dataset_train.csv"))
    val_df = pd.read_csv(os.path.join(BASE_DIR, "dataset_val.csv"))
    train_df = apply_obfuscation_labels(train_df, obf_labeler)
    val_df = apply_obfuscation_labels(val_df, obf_labeler)

    obf_train = train_df.set_index("filename")["is_obfuscated"].to_dict()
    obf_val = val_df.set_index("filename")["is_obfuscated"].to_dict()

    train_ds = make_dataset("train", obf_train, csv_scaler, shuffle=True, cache=True)
    val_ds = make_dataset("val", obf_val, csv_scaler, shuffle=False, cache=True)

    print("Caricamento CNN già addestrata...")
    best_model_path = os.path.join(CNN_MODEL_DIR, "best_model.keras")
    if not os.path.isfile(best_model_path):
        raise FileNotFoundError(f"\nCNN non trovata in: {best_model_path}\nAddestra prima il CNN (sezione 3) prima di lanciare main_train_vae().")
    cnn_model = load_trained_cnn(best_model_path)

    print("Estrazione embedding train/val...")
    train_emb, _, _ = extract_fusion_embeddings(cnn_model, train_ds)
    val_emb, _, y_obf_val = extract_fusion_embeddings(cnn_model, val_ds)

    vae, train_losses, val_losses = train_vae_on_embeddings(train_emb, val_emb)

    vae_weights_dir = VAE_MODEL_DIR
    os.makedirs(vae_weights_dir, exist_ok=True)
    vae.encoder.save(os.path.join(vae_weights_dir, "vae_encoder.keras"))
    vae.decoder.save(os.path.join(vae_weights_dir, "vae_decoder.keras"))
    print(f"VAE salvato in: {vae_weights_dir}")

    print("\nCalibrazione reconstruction error sul train...")
    train_errors = compute_vae_reconstruction_errors(vae, train_emb)
    err_min = float(train_errors.min())
    err_max = float(train_errors.max())

    calib_path = os.path.join(VAE_MODEL_DIR, "vae_calibration.json")
    with open(calib_path, "w") as f:
        json.dump({
            "err_min": err_min,
            "err_max": err_max,
            "embedding_source": "fusion_bn",
            "latent_dim": VAE_LATENT_DIM
        }, f)
    print(f"Calibrazione salvata: {calib_path}")

    print("\nValutazione su validation set...")
    val_errors = compute_vae_reconstruction_errors(vae, val_emb)
    val_pct = errors_to_percentage(val_errors, err_min, err_max)

    try:
        vae_auc = roc_auc_score(y_obf_val, val_errors)
        print(f"VAE ROC-AUC (val, reconstruction error vs is_obfuscated): {vae_auc:.4f}")
    except Exception as e:
        print(f"Impossibile calcolare VAE ROC-AUC: {e}")

    plot_vae_results(train_losses, val_losses, val_errors, y_obf_val, val_pct, VAE_TRAIN_DIR)

    elapsed_total = (time.time() - t_start) / 60
    print("\n" + "=" * 70)
    print(f"VAE TRAINING COMPLETATO in {elapsed_total:.1f} minuti")
    print(f"Modello: {VAE_MODEL_DIR}")
    print(f"Plot training: {VAE_TRAIN_DIR}")
    print("=" * 70)

    return vae

vae_model = main_train_vae()

**Test VAE.** Richiede CNN e VAE già addestrati (encoder/decoder in `VAE_MODEL_DIR`). Valuta sul test set, produce grafici specifici VAE in `VAE_TEST_DIR` e un CSV con la percentuale di offuscamento stimata per ogni sample di test.

In [ ]:
# ==============================================================
# SEZIONE 5b — VAE: TEST
# ==============================================================

def main_test_vae():
    t_start = time.time()

    print("\n" + "=" * 70)
    print("TEST — CNN + VAE (embedding)")
    print("=" * 70)

    csv_scaler_path = os.path.join(CNN_MODEL_DIR, "csv_scaler.pkl")
    obf_labeler_path = os.path.join(CNN_MODEL_DIR, "obf_labeler.pkl")

    csv_scaler = joblib.load(csv_scaler_path)
    obf_labeler = joblib.load(obf_labeler_path)

    print("Caricamento e labeling test set...")
    test_df = pd.read_csv(os.path.join(BASE_DIR, "dataset_test.csv"))
    test_df = apply_obfuscation_labels(test_df, obf_labeler)
    obf_test = test_df.set_index("filename")["is_obfuscated"].to_dict()

    test_ds = make_dataset("test", obf_test, csv_scaler, shuffle=False, cache=True)

    print("\n" + "-" * 70)
    print("CNN — VALUTAZIONE")
    print("-" * 70)

    best_model_path = os.path.join(CNN_MODEL_DIR, "best_model.keras")
    cnn_model = load_trained_cnn(best_model_path)

    res = get_predictions(cnn_model, test_ds)
    print_classification_report(res)
    save_plots([], [], res, output_dir=CNN_TEST_DIR)

    print("\n" + "-" * 70)
    print("VAE — VALUTAZIONE")
    print("-" * 70)

    vae_dir = VAE_MODEL_DIR
    encoder_path = os.path.join(vae_dir, "vae_encoder.keras")
    decoder_path = os.path.join(vae_dir, "vae_decoder.keras")
    calib_path = os.path.join(VAE_MODEL_DIR, "vae_calibration.json")

    if not os.path.isfile(encoder_path) or not os.path.isfile(decoder_path) or not os.path.isfile(calib_path):
        raise FileNotFoundError("\nVAE o calibrazione non trovati. Esegui prima main_train_vae() (sezione 5).")

    encoder = tf.keras.models.load_model(encoder_path, compile=False, custom_objects={"Sampling": Sampling})
    decoder = tf.keras.models.load_model(decoder_path, compile=False)
    vae_model = VAE(encoder, decoder)

    with open(calib_path) as f:
        calib = json.load(f)

    print("Estrazione embedding test...")
    test_emb, _, y_obf_test = extract_fusion_embeddings(cnn_model, test_ds)

    print("Calcolo reconstruction error...")
    test_errors = compute_vae_reconstruction_errors(vae_model, test_emb)
    test_pct = errors_to_percentage(test_errors, calib["err_min"], calib["err_max"])

    try:
        vae_auc = roc_auc_score(y_obf_test, test_errors)
        print(f"VAE ROC-AUC (reconstruction error vs is_obfuscated): {vae_auc:.4f}")
    except Exception as e:
        print(f"Impossibile calcolare VAE ROC-AUC: {e}")

    plot_vae_results([], [], test_errors, y_obf_test, test_pct, VAE_TEST_DIR)

    print("Lettura filename del test TFRecord...")
    filenames = []
    test_path = os.path.join(TFRECORD_DIR, "test.tfrecord")

    for proto in tf.data.TFRecordDataset(test_path):
        p = tf.io.parse_single_example(proto, {"filename": tf.io.FixedLenFeature([], tf.string)})
        filenames.append(p["filename"].numpy().decode())

    n = min(len(filenames), len(test_pct))

    out_df = pd.DataFrame({
        "filename": filenames[:n],
        "obf_pct_unsupervised": test_pct[:n]
    })

    csv_out = os.path.join(VAE_TEST_DIR, "obfuscation_percentage_test_vae.csv")
    out_df.to_csv(csv_out, index=False)

    elapsed_total = (time.time() - t_start) / 60
    print("\n" + "=" * 70)
    print(f"TEST VAE COMPLETATO in {elapsed_total:.1f} minuti")
    print(f"Output: {VAE_TEST_DIR}")
    print(f"CSV obfuscation: {csv_out}")
    print("=" * 70)

main_test_vae()

---
## 6. Riepilogo finale — training e test di tutti i modelli

Cella unica che esegue in sequenza **training e test** di CNN, Isolation Forest e VAE, richiamando le funzioni `main_*` già definite nelle sezioni precedenti. Utile per lanciare l'intera pipeline in un colpo solo dopo aver preparato i TFRecord (sezione 1) ed eseguito la base condivisa (sezione 2).

Al termine stampa un riepilogo con i percorsi di tutte le directory `model/`, `train/` e `test/` popolate per ciascun modello.

**Richiede:** sezione 1 (TFRecord) e sezione 2 (BASE CONDIVISA) già eseguite in questa sessione. Le sezioni 3, 4, 5 non serve eseguirle a parte: questa cella richiama direttamente `main_train_cnn`, `main_test_cnn`, `main_train_if`, `main_test_if`, `main_train_vae`, `main_test_vae` definite sopra — ma quelle celle vanno comunque eseguite prima, perché è lì che tali funzioni vengono definite.

In [ ]:
# ==============================================================
# SEZIONE 6 — RIEPILOGO FINALE: TRAIN + TEST DI TUTTI I MODELLI
# Richiede la sezione 2 (BASE CONDIVISA) già eseguita.
# Esegue in sequenza: training e test di CNN, poi Isolation
# Forest, poi VAE. Utile per lanciare l'intera pipeline con
# un'unica cella dopo aver preparato i TFRecord.
# ==============================================================

def run_full_pipeline():
    t_start = time.time()

    print("\n" + "#" * 70)
    print("# PIPELINE COMPLETA — CNN + ISOLATION FOREST + VAE")
    print("#" * 70)

    # ---------------- CNN ----------------
    print("\n" + "#" * 70)
    print("# 1/6 — TRAINING CNN")
    print("#" * 70)
    cnn_model = main_train_cnn()

    print("\n" + "#" * 70)
    print("# 2/6 — TEST CNN")
    print("#" * 70)
    cnn_model = main_test_cnn()

    # ---------------- ISOLATION FOREST ----------------
    print("\n" + "#" * 70)
    print("# 3/6 — TRAINING ISOLATION FOREST")
    print("#" * 70)
    iso_model = main_train_if()

    print("\n" + "#" * 70)
    print("# 4/6 — TEST ISOLATION FOREST")
    print("#" * 70)
    main_test_if()

    # ---------------- VAE ----------------
    print("\n" + "#" * 70)
    print("# 5/6 — TRAINING VAE")
    print("#" * 70)
    vae_model = main_train_vae()

    print("\n" + "#" * 70)
    print("# 6/6 — TEST VAE")
    print("#" * 70)
    main_test_vae()

    elapsed_total = (time.time() - t_start) / 60
    print("\n" + "#" * 70)
    print(f"PIPELINE COMPLETA TERMINATA in {elapsed_total:.1f} minuti")
    print("#" * 70)
    print(f"CNN  → modello: {CNN_MODEL_DIR} | train: {CNN_TRAIN_DIR} | test: {CNN_TEST_DIR}")
    print(f"IF   → modello: {IF_MODEL_DIR} | train: {IF_TRAIN_DIR} | test: {IF_TEST_DIR}")
    print(f"VAE  → modello: {VAE_MODEL_DIR} | train: {VAE_TRAIN_DIR} | test: {VAE_TEST_DIR}")
    print("#" * 70)

    return cnn_model, iso_model, vae_model


cnn_model, iso_model, vae_model = run_full_pipeline()
